In [14]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "id": "775a78a0",
   "metadata": {},
   "source": [
    "# Lab 1 - Microstates of a system of coins\n",
    "\n",
    "Runs the code in `src/microstates.py`, its unit tests, and `main.py`. Use **Run All** in VS Code or Jupyter Notebook."
   ]
  },
  {
   "cell_type": "markdown",
   "id": "95f76ecd",
   "metadata": {},
   "source": [
    "## Setup\n",
    "Adds the repository root directory to Python's `sys.path` so that modules inside `src/` can be imported seamlessly."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 1,
   "id": "66e324e4",
   "metadata": {},
   "outputs": [],
   "source": [
    "import os\n",
    "import sys\n",
    "import subprocess\n",
    "from math import comb\n",
    "from pathlib import Path\n",
    "\n",
    "ROOT = Path.cwd() if (Path.cwd() / \"src\").exists() else Path.cwd().parent\n",
    "sys.path.insert(0, str(ROOT))\n",
    "\n",
    "import matplotlib.pyplot as plt\n",
    "import numpy as np\n",
    "\n",
    "from src.microstates import (\n",
    "    generate_microstates_baseline,\n",
    "    generate_microstates_bitwise,\n",
    "    measure_microstates_optimized,\n",
    ")\n",
    "\n",
    "print(\"Repo root:\", ROOT)"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "86fa341b",
   "metadata": {},
   "source": [
    "## Unit Tests\n",
    "Executes `pytest` on the `tests/` directory to verify all test cases pass."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 2,
   "id": "472910db",
   "metadata": {},
   "outputs": [],
   "source": [
    "result = subprocess.run([sys.executable, \"-m\", \"pytest\", \"-v\"], cwd=ROOT, capture_output=True, text=True)\n",
    "print(result.stdout if result.stdout else result.stderr)"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "5e68ccb8",
   "metadata": {},
   "source": [
    "## Bitwise Encoding Mechanics\n",
    "Each integer $s \\in [0, 2^N)$ represents one unique microstate. Bit $i$ indicates coin $i$ ($1 = \\text{Heads}$, $0 = \\text{Tails}$)."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 3,
   "id": "f869bbbd",
   "metadata": {},
   "outputs": [],
   "source": [
    "N = 3\n",
    "for s in range(1 << N):\n",
    "    binary_str = f\"{s:0{N}b}\"\n",
    "    coins = \"\".join(['H' if bit == '1' else 'T' for bit in binary_str])\n",
    "    heads = bin(s).count('1')\n",
    "    print(f\"s = {s}  binary = {binary_str}  coins = {coins}  heads = {heads}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "3c54ec4c",
   "metadata": {},
   "source": [
    "## 1a: Microstate Breakdown for $N = 7$ Coins ($2^7 = 128$ states)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 4,
   "id": "5a4944e8",
   "metadata": {},
   "outputs": [],
   "source": [
    "N = 7\n",
    "groups = {}\n",
    "for s in range(1 << N):\n",
    "    heads = bin(s).count('1')\n",
    "    groups.setdefault(heads, []).append(s)\n",
    "\n",
    "print(f\"{'macrostate':<22}{'# microstates':>14}\")\n",
    "for k in sorted(groups.keys(), reverse=True):\n",
    "    label = f\"{k} heads, {N-k} tails\"\n",
    "    print(f\"{label:<22}{len(groups[k]):>14}\")\n",
    "print(f\"{'total':<22}{sum(len(v) for v in groups.values()):>14}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "fe261747",
   "metadata": {},
   "source": [
    "## Verification: Binomial Coefficients $\\Omega(N,k) = \\binom{N}{k}$"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 5,
   "id": "39f70269",
   "metadata": {},
   "outputs": [],
   "source": [
    "for N in [5, 7, 10, 16]:\n",
    "    counts = [0] * (N + 1)\n",
    "    for s in range(1 << N):\n",
    "        counts[bin(s).count('1')] += 1\n",
    "    expected = [comb(N, k) for k in range(N + 1)]\n",
    "    print(f\"N = {N:2d}: {counts}  matches comb: {counts == expected}  sum = 2^{N}: {sum(counts) == 2**N}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "efce0096",
   "metadata": {},
   "source": [
    "## 1b: Performance Benchmarking Across Methods"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 6,
   "id": "863255fd",
   "metadata": {},
   "outputs": [],
   "source": [
    "print(f\"{'N':<4} | {'2^N Microstates':<16} | {'Method':<22} | {'Time (s)':<12} | {'RAM (MiB)':<10}\")\n",
    "print(\"-\" * 72)\n",
    "\n",
    "methods = [\n",
    "    (\"Method 1 (Itertools)\", generate_microstates_baseline),\n",
    "    (\"Method 2 (Bitwise Gen)\", generate_microstates_bitwise),\n",
    "    (\"Method 3 (Optimized)\", measure_microstates_optimized),\n",
    "]\n",
    "\n",
    "for N in range(10, 22, 4):\n",
    "    for name, func in methods:\n",
    "        count, exec_time, ram_mb = func(N)\n",
    "        print(f\"{N:<4} | {count:<16} | {name:<22} | {exec_time:<12.6f} | {ram_mb:<10.4f}\")\n",
    "    print(\"-\" * 72)"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "331229fa",
   "metadata": {},
   "source": [
    "## 1b: Time & Memory Scaling Analysis"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 7,
   "id": "7339f285",
   "metadata": {},
   "outputs": [],
   "source": [
    "ns, times, rams = [], [], []\n",
    "for N in range(10, 26, 2):\n",
    "    count, exec_time, ram_mb = measure_microstates_optimized(N)\n",
    "    ns.append(N)\n",
    "    times.append(exec_time)\n",
    "    rams.append(ram_mb)\n",
    "\n",
    "fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))\n",
    "ax1.plot(ns, times, \"o-\", color=\"tab:blue\")\n",
    "ax1.set_yscale(\"log\")\n",
    "ax1.set_xlabel(\"Number of coins $N$\")\n",
    "ax1.set_ylabel(\"Execution Time (s)\")\n",
    "ax1.set_title(\"Log Execution Time vs N\")\n",
    "ax1.grid(True, linestyle=\"--\", alpha=0.6)\n",
    "\n",
    "ax2.plot(ns, rams, \"s-\", color=\"tab:red\")\n",
    "ax2.set_xlabel(\"Number of coins $N$\")\n",
    "ax2.set_ylabel(\"Peak RAM (MiB)\")\n",
    "ax2.set_title(\"Memory Footprint vs N\")\n",
    "ax2.grid(True, linestyle=\"--\", alpha=0.6)\n",
    "\n",
    "fig.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "746cc59f",
   "metadata": {},
   "source": [
    "## Execution of `main.py`\n",
    "Runs the main project entry point and displays standard output."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": 8,
   "id": "f3d381a5",
   "metadata": {},
   "outputs": [],
   "source": [
    "result = subprocess.run([sys.executable, \"main.py\"], cwd=ROOT, capture_output=True, text=True)\n",
    "print(result.stdout if result.stdout else result.stderr)"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "statphys_lab1",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "codemirror_mode": {
    "name": "ipython",
    "version": 3
   },
   "file_extension": ".py",
   "mimetype": "text/x-python",
   "name": "python",
   "nbconvert_exporter": "python",
   "pygments_lexer": "ipython3",
   "version": "3.11.17"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 5
}

{'cells': [{'cell_type': 'markdown',
   'id': '775a78a0',
   'metadata': {},
   'source': ['# Lab 1 - Microstates of a system of coins\n',
    '\n',
    'Runs the code in `src/microstates.py`, its unit tests, and `main.py`. Use **Run All** in VS Code or Jupyter Notebook.']},
  {'cell_type': 'markdown',
   'id': '95f76ecd',
   'metadata': {},
   'source': ['## Setup\n',
    "Adds the repository root directory to Python's `sys.path` so that modules inside `src/` can be imported seamlessly."]},
  {'cell_type': 'code',
   'execution_count': 1,
   'id': '66e324e4',
   'metadata': {},
   'outputs': [],
   'source': ['import os\n',
    'import sys\n',
    'import subprocess\n',
    'from math import comb\n',
    'from pathlib import Path\n',
    '\n',
    'ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent\n',
    'sys.path.insert(0, str(ROOT))\n',
    '\n',
    'import matplotlib.pyplot as plt\n',
    'import numpy as np\n',
    '\n',
    'from src.microstates import (